что я делаю
загружаю все 3 матрицы
объединяю их в одну итоговую матрицу user item
выделяю релевантных пользователей
уже на этой общей матрице запускаю candidate generation и ranking

Важно
сильное прослушивание played ratio 80 считаем положительным сигналом
слабое прослушивание считаем нейтральным слабым сигналом
если по паре есть дизлайк он имеет приоритет над положительным сигналом


In [25]:
import os
from pathlib import Path

os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")

import numpy as np
import pandas as pd
from IPython.display import display
from implicit.als import AlternatingLeastSquares
from scipy.sparse import csr_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import NearestNeighbors
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

In [26]:
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 180)

PROJECT_DIR = Path.cwd().resolve()
if not (PROJECT_DIR / "checkpoint-5-Edgar").exists():
    PROJECT_DIR = PROJECT_DIR.parent

INTERACTION_DIR = PROJECT_DIR / "checkpoint-5-Edgar" / "interaction"
LIKES_DIR = INTERACTION_DIR / "likes"
DISLIKES_DIR = INTERACTION_DIR / "dislikes"
LISTENS_DIR = INTERACTION_DIR / "listens"

USER_ACTIVITY_LOW_Q = 0.70
USER_ACTIVITY_HIGH_Q = 0.95
MIN_POSITIVE_INTERACTIONS = 2

LISTEN_POSITIVE_THRESHOLD = 0.80
LISTEN_WEIGHT_COEF = 0.30

RANDOM_STATE = 42
CANDIDATE_K = 100
FINAL_K = 10

FINAL_MATRIX_PATH = INTERACTION_DIR / "final_interaction_matrix_relevant_users.csv"

LIKES_DIR, DISLIKES_DIR, LISTENS_DIR


(WindowsPath('D:/Projects/PyCharm/hse-ai-year-project-2025/checkpoint-5-Edgar/interaction/likes'),
 WindowsPath('D:/Projects/PyCharm/hse-ai-year-project-2025/checkpoint-5-Edgar/interaction/dislikes'),
 WindowsPath('D:/Projects/PyCharm/hse-ai-year-project-2025/checkpoint-5-Edgar/interaction/listens'))

берём уже готовые csv от работы Богдана
likes dislikes listens


In [27]:
def read_pair_tables(source_dir, source_name):
    sign_path = source_dir / f"user_item_sign_{source_name}.csv"
    weight_path = source_dir / f"user_item_weighted_{source_name}.csv"

    sign_df = pd.read_csv(sign_path).drop(columns=["Unnamed: 0"], errors="ignore")
    weight_df = pd.read_csv(weight_path).drop(columns=["Unnamed: 0"], errors="ignore")

    frame = sign_df.merge(weight_df, on=["uid", "item_id"], how="inner")
    frame["source"] = source_name
    return frame

likes_with_old_negatives = read_pair_tables(LIKES_DIR, "likes")
dislikes_raw = read_pair_tables(DISLIKES_DIR, "dislikes")
listens_raw = read_pair_tables(LISTENS_DIR, "listens")

likes_raw = likes_with_old_negatives[likes_with_old_negatives["interaction_sign"] == 1].copy()

source_overview = pd.DataFrame(
    [
        {
            "source": "likes (после очистки от старых -1 строк)",
            "rows": len(likes_raw),
            "users": likes_raw["uid"].nunique(),
            "items": likes_raw["item_id"].nunique(),
        },
        {
            "source": "dislikes",
            "rows": len(dislikes_raw),
            "users": dislikes_raw["uid"].nunique(),
            "items": dislikes_raw["item_id"].nunique(),
        },
        {
            "source": "listens",
            "rows": len(listens_raw),
            "users": listens_raw["uid"].nunique(),
            "items": listens_raw["item_id"].nunique(),
        },
    ]
)

display(source_overview)


,source,rows,users,items
0,likes (после очистки от старых -1 строк),161153,6850,68336
1,dislikes,16703,2983,12298
2,listens,2427547,8631,388631


перед объединением полезно проверить сколько отрицательных строк было внутри старого массива лайков
насколько пересекаются лайки и дизлайки
как распределён вес у прослушиваний


In [28]:
likes_negative_rows = int((likes_with_old_negatives["interaction_sign"] == -1).sum())

like_dislike_overlap = (
    likes_raw[["uid", "item_id"]]
    .merge(dislikes_raw[["uid", "item_id"]], on=["uid", "item_id"], how="inner")
)

print(f"отрицательных строк внутри старого likes-файла: {likes_negative_rows:,}")
print(f"после очистки likes осталось строк: {len(likes_raw):,}")
print(f"пересечение пар между likes и dislikes: {len(like_dislike_overlap):,}")
print()
print("распределение веса в listens:")
display(
    listens_raw["interaction_weighted"]
    .describe(percentiles=[0.25, 0.50, 0.75, 0.80, 0.90, 0.95])
    .to_frame("listen_weight")
)


отрицательных строк внутри старого likes-файла: 16,252
после очистки likes осталось строк: 161,153
пересечение пар между likes и dislikes: 452

распределение веса в listens:


,listen_weight
count,2.427547e+06
mean,5.562689e-01
std,4.533582e-01
min,0.000000e+00
25%,3.000000e-02
50%,7.400000e-01
75%,1.000000e+00
80%,1.000000e+00
90%,1.000000e+00
95%,1.000000e+00


Здесь уже появляется общая матрица интеракций
По каждой паре uid item id храним:
был ли лайк,
был ли дизлайк,
было ли прослушивание,
итоговый знак взаимодействия,
итоговый вес взаимодействия


подход для матрицы интеракций:
дизлайк доминирует.
если дизлайка нет, лайк считается сильным позитивом.
если лайка нет, но есть сильное прослушивание (80%) считаем это позитивом
слабое прослушивание оставляем нейтральным слабым сигналом


In [29]:
likes_pairs = likes_raw[["uid", "item_id", "interaction_weighted"]].copy()
likes_pairs = likes_pairs.rename(columns={"interaction_weighted": "like_weight"})
likes_pairs["has_like"] = 1

dislikes_pairs = dislikes_raw[["uid", "item_id", "interaction_weighted"]].copy()
dislikes_pairs = dislikes_pairs.rename(columns={"interaction_weighted": "dislike_weight"})
dislikes_pairs["has_dislike"] = 1

listens_pairs = listens_raw[["uid", "item_id", "interaction_weighted"]].copy()
listens_pairs = listens_pairs.rename(columns={"interaction_weighted": "listen_weight"})
listens_pairs["has_listen"] = 1

interaction_pairs = likes_pairs.merge(dislikes_pairs, on=["uid", "item_id"], how="outer")
interaction_pairs = interaction_pairs.merge(listens_pairs, on=["uid", "item_id"], how="outer")

interaction_pairs["uid"] = interaction_pairs["uid"].astype("int32")
interaction_pairs["item_id"] = interaction_pairs["item_id"].astype("int32")

for col in ["has_like", "has_dislike", "has_listen"]:
    interaction_pairs[col] = interaction_pairs[col].fillna(0).astype("int8")

for col in ["like_weight", "dislike_weight", "listen_weight"]:
    interaction_pairs[col] = interaction_pairs[col].fillna(0.0).astype("float32")

interaction_pairs["strong_listen"] = (
    interaction_pairs["listen_weight"] >= LISTEN_POSITIVE_THRESHOLD
).astype("int8")

interaction_pairs["is_negative"] = interaction_pairs["has_dislike"].astype("int8")
interaction_pairs["is_positive"] = (
    (interaction_pairs["has_dislike"] == 0)
    & (
        (interaction_pairs["has_like"] == 1)
        | (interaction_pairs["strong_listen"] == 1)
    )
).astype("int8")
interaction_pairs["is_neutral"] = (
    (interaction_pairs["is_positive"] == 0)
    & (interaction_pairs["is_negative"] == 0)
).astype("int8")

interaction_pairs["interaction_sign"] = np.select(
    [
        interaction_pairs["is_negative"] == 1,
        interaction_pairs["is_positive"] == 1,
    ],
    [-1, 1],
    default=0,
).astype("int8")

positive_confidence = (
    interaction_pairs["like_weight"] + LISTEN_WEIGHT_COEF * interaction_pairs["listen_weight"]
).astype("float32")

interaction_pairs["interaction_weighted"] = np.where(
    interaction_pairs["is_negative"] == 1,
    interaction_pairs["dislike_weight"].clip(lower=0.1),
    positive_confidence,
).astype("float32")
interaction_pairs["interaction_weighted"] = interaction_pairs["interaction_weighted"].clip(lower=0.05, upper=1.30)

signal_case_summary = (
    interaction_pairs.groupby(["has_like", "has_dislike", "has_listen"], as_index=False)
    .size()
    .sort_values("size", ascending=False)
)

display(signal_case_summary.head(10))
interaction_pairs.head()


,has_like,has_dislike,has_listen,size
0,0,0,1,2320019
4,1,0,1,100244
3,1,0,0,60457
1,0,1,0,9307
2,0,1,1,6944
6,1,1,1,340
5,1,1,0,112


,uid,item_id,like_weight,has_like,dislike_weight,has_dislike,listen_weight,has_listen,strong_listen,is_negative,is_positive,is_neutral,interaction_sign,interaction_weighted
0,100,88111,0.0,0,0.0,0,1.0,1,1,0,1,0,1,0.3
1,100,152884,0.0,0,0.0,0,1.0,1,1,0,1,0,1,0.3
2,100,155792,0.0,0,0.0,0,1.0,1,1,0,1,0,1,0.3
3,100,225669,0.0,0,0.0,0,1.0,1,1,0,1,0,1,0.3
4,100,286361,0.0,0,0.0,0,1.0,1,1,0,1,0,1,0.3


теперь нам нужна не просто любая матрица а матрица для релевантных пользователей

здесь используем простую логику
берём пользователей между 70 и 95 перцентилями по числу пар
требуем минимум 2 положительных сигнала

зачем так делать ?
слишком холодные пользователи плохо подходят для baseline
слишком активные могут сильно перекосить картину


In [30]:
user_profile_stats = interaction_pairs.groupby("uid", as_index=False).agg(
    total_pairs=("item_id", "nunique"),
    positive_pairs=("is_positive", "sum"),
    negative_pairs=("is_negative", "sum"),
    neutral_pairs=("is_neutral", "sum"),
    like_pairs=("has_like", "sum"),
    dislike_pairs=("has_dislike", "sum"),
    listen_pairs=("has_listen", "sum"),
    strong_listen_pairs=("strong_listen", "sum"),
    avg_interaction_weight=("interaction_weighted", "mean"),
)

low_threshold = user_profile_stats["total_pairs"].quantile(USER_ACTIVITY_LOW_Q)
high_threshold = user_profile_stats["total_pairs"].quantile(USER_ACTIVITY_HIGH_Q)

user_profile_stats["is_relevant_user"] = (
    user_profile_stats["total_pairs"].between(low_threshold, high_threshold, inclusive="both")
    & (user_profile_stats["positive_pairs"] >= MIN_POSITIVE_INTERACTIONS)
).astype("int8")

relevant_user_ids = set(
    user_profile_stats.loc[user_profile_stats["is_relevant_user"] == 1, "uid"].astype(int)
)

relevant_user_stats = user_profile_stats[user_profile_stats["is_relevant_user"] == 1].copy()

interaction_matrix = (
    interaction_pairs[interaction_pairs["uid"].isin(relevant_user_ids)]
    .sort_values(["uid", "item_id"])
    .reset_index(drop=True)
    .copy()
)

interaction_matrix.to_csv(FINAL_MATRIX_PATH, index=False)

pd.DataFrame(
    [
        {
            "low_threshold": low_threshold,
            "high_threshold": high_threshold,
            "relevant_users": len(relevant_user_ids),
            "saved_to": str(FINAL_MATRIX_PATH),
        }
    ]
)


,low_threshold,high_threshold,relevant_users,saved_to
0,313.0,820.95,2366,D:\Projects\PyCharm\hse-ai-year-project-2025\c...


Это и есть главный результат подготовки данных
одна общая матрица user item в которой уже сидят:
лайки,
дизлайки,
прослушивания,
итоговые флаги positive negative neutral,
только релевантные пользователи


In [31]:
positive_pairs = interaction_matrix[interaction_matrix["is_positive"] == 1].copy()
negative_pairs = interaction_matrix[interaction_matrix["is_negative"] == 1].copy()
neutral_pairs = interaction_matrix[interaction_matrix["is_neutral"] == 1].copy()

matrix_summary = pd.Series(
    {
        "строк в итоговой матрице": len(interaction_matrix),
        "релевантных пользователей": interaction_matrix["uid"].nunique(),
        "уникальных айтемов": interaction_matrix["item_id"].nunique(),
        "положительных пар": int(positive_pairs["is_positive"].sum()),
        "отрицательных пар": int(negative_pairs["is_negative"].sum()),
        "нейтральных пар": int(neutral_pairs["is_neutral"].sum()),
    }
)

display(matrix_summary.to_frame("value"))
display(
    relevant_user_stats[
        ["total_pairs", "positive_pairs", "negative_pairs", "listen_pairs", "strong_listen_pairs"]
    ].describe().round(2)
)
interaction_matrix.head()


,value
строк в итоговой матрице,1175485
релевантных пользователей,2366
уникальных айтемов,252758
положительных пар,610698
отрицательных пар,6355
нейтральных пар,558432


,total_pairs,positive_pairs,negative_pairs,listen_pairs,strong_listen_pairs
count,2366.00,2366.00,2366.00,2366.00,2366.00
mean,496.82,258.11,2.69,490.54,244.10
std,135.73,134.38,17.09,140.29,136.59
min,313.00,6.00,0.00,0.00,0.00
25%,380.00,159.00,0.00,375.00,144.00
50%,471.00,242.50,0.00,467.00,229.00
75%,590.00,335.00,2.00,585.00,324.00
max,820.00,777.00,627.00,820.00,738.00


,uid,item_id,like_weight,has_like,dislike_weight,has_dislike,listen_weight,has_listen,strong_listen,is_negative,is_positive,is_neutral,interaction_sign,interaction_weighted
0,700,3957,0.0,0,0.0,0,1.00,1,1,0,1,0,1,0.300
1,700,48074,0.0,0,0.0,0,0.09,1,0,0,0,1,0,0.050
2,700,67412,0.0,0,0.0,0,0.99,1,1,0,1,0,1,0.297
3,700,90866,0.0,0,0.0,0,0.26,1,0,0,0,1,0,0.078
4,700,122944,0.0,0,0.0,0,0.04,1,0,0,0,1,0,0.050


положительные сигналы делим на train validation test
отрицательные и нейтральные пары целиком оставляем в history

положительный сигнал здесь это
лайк
или сильное прослушивание без дизлайка


In [32]:
def split_positive_pairs(positive_df, random_state=42):
    rng = np.random.default_rng(random_state)

    train_parts = []
    val_parts = []
    test_parts = []

    for uid, group in positive_df.groupby("uid"):
        group = group.sample(frac=1.0, random_state=int(rng.integers(0, 1_000_000_000)))
        n = len(group)

        if n == 1:
            train_parts.append(group)
        elif n == 2:
            test_parts.append(group.iloc[:1].copy())
            train_parts.append(group.iloc[1:].copy())
        else:
            test_parts.append(group.iloc[:1].copy())
            val_parts.append(group.iloc[1:2].copy())
            train_parts.append(group.iloc[2:].copy())

    train_pos = pd.concat(train_parts, ignore_index=True) if train_parts else positive_df.iloc[0:0].copy()
    val_pos = pd.concat(val_parts, ignore_index=True) if val_parts else positive_df.iloc[0:0].copy()
    test_pos = pd.concat(test_parts, ignore_index=True) if test_parts else positive_df.iloc[0:0].copy()
    return train_pos, val_pos, test_pos

train_pos, val_pos, test_pos = split_positive_pairs(positive_pairs, random_state=RANDOM_STATE)

pd.DataFrame(
    [
        {"split": "train_pos", "rows": len(train_pos), "users": train_pos["uid"].nunique()},
        {"split": "val_pos", "rows": len(val_pos), "users": val_pos["uid"].nunique()},
        {"split": "test_pos", "rows": len(test_pos), "users": test_pos["uid"].nunique()},
        {"split": "negative_pairs", "rows": len(negative_pairs), "users": negative_pairs["uid"].nunique()},
    ]
)

,split,rows,users
0,train_pos,605966,2366
1,val_pos,2366,2366
2,test_pos,2366,2366
3,negative_pairs,6355,1041


логика теперь такая:

на валидации модель учится на train данных: берём позитивные, негативные и нейтральные пары.
Перед тестом дообучаемся уже на train + val позитиве (плюс те же negative и neutral).

то есть слабые прослушивания не становятся target но остаются в истории как контекст


In [33]:
train_pos, val_pos, test_pos = split_positive_pairs(positive_pairs, random_state=RANDOM_STATE)

val_history_all = pd.concat([train_pos, negative_pairs, neutral_pairs], ignore_index=True)
test_history_all = pd.concat([train_pos, val_pos, negative_pairs, neutral_pairs], ignore_index=True)

pd.DataFrame(
    [
        {"part": "train_pos", "rows": len(train_pos)},
        {"part": "val_pos", "rows": len(val_pos)},
        {"part": "test_pos", "rows": len(test_pos)},
        {"part": "negative_pairs", "rows": len(negative_pairs)},
        {"part": "neutral_pairs", "rows": len(neutral_pairs)},
        {"part": "val_history_all", "rows": len(val_history_all)},
        {"part": "test_history_all", "rows": len(test_history_all)},
    ]
)


,part,rows
0,train_pos,605966
1,val_pos,2366
2,test_pos,2366
3,negative_pairs,6355
4,neutral_pairs,558432
5,val_history_all,1170753
6,test_history_all,1173119


здесь происходит сразу несколько вещей:
positive матрица для popular и userknn,
weighted матрица для als,
truth для оценки,
набор уже увиденных айтемов,

теперь positive матрица строится уже не только из лайков
а из лайков сильных прослушиваний


In [34]:
def build_stage(history_all_df, target_pos_df):
    history_positive = history_all_df[history_all_df["is_positive"] == 1].copy()

    user_ids = np.sort(history_positive["uid"].unique())
    item_ids = np.sort(history_positive["item_id"].unique())

    user2idx = {int(uid): idx for idx, uid in enumerate(user_ids)}
    item2idx = {int(item_id): idx for idx, item_id in enumerate(item_ids)}

    history_positive = history_positive[
        history_positive["uid"].isin(user_ids) & history_positive["item_id"].isin(item_ids)
    ].copy()
    history_all_df = history_all_df[
        history_all_df["uid"].isin(user_ids) & history_all_df["item_id"].isin(item_ids)
    ].copy()

    history_positive["user_idx"] = history_positive["uid"].map(user2idx).astype("int32")
    history_positive["item_idx"] = history_positive["item_id"].map(item2idx).astype("int32")

    history_all_df["user_idx"] = history_all_df["uid"].map(user2idx).astype("int32")
    history_all_df["item_idx"] = history_all_df["item_id"].map(item2idx).astype("int32")

    shape = (len(user_ids), len(item_ids))

    binary_matrix = csr_matrix(
        (
            np.ones(len(history_positive), dtype=np.float32),
            (history_positive["user_idx"], history_positive["item_idx"]),
        ),
        shape=shape,
    )

    weighted_matrix = csr_matrix(
        (
            history_all_df["interaction_weighted"].astype("float32"),
            (history_all_df["user_idx"], history_all_df["item_idx"]),
        ),
        shape=shape,
    )

    target_pos_df = target_pos_df[
        target_pos_df["uid"].isin(user_ids) & target_pos_df["item_id"].isin(item_ids)
    ].copy()

    truth = (
        target_pos_df[["uid", "item_id"]]
        .drop_duplicates()
        .groupby("uid")["item_id"]
        .agg(set)
        .to_dict()
    )

    seen_items = {
        int(uid): group["item_idx"].to_numpy(dtype=np.int32, copy=True)
        for uid, group in history_all_df.groupby("uid")
    }

    negative_sets = (
        history_all_df[history_all_df["is_negative"] == 1]
        .groupby("uid")["item_id"]
        .agg(set)
        .to_dict()
    )

    return {
        "history_all": history_all_df,
        "history_positive": history_positive,
        "truth": truth,
        "user2idx": user2idx,
        "item2idx": item2idx,
        "idx2item": item_ids,
        "binary_matrix": binary_matrix.tocsr(),
        "weighted_matrix": weighted_matrix.tocsr(),
        "seen_items": seen_items,
        "negative_sets": negative_sets,
    }

val_stage = build_stage(val_history_all, val_pos)
test_stage = build_stage(test_history_all, test_pos)

это быстрый чек перед моделями


In [35]:
pd.DataFrame(
    [
        {
            "stage": "validation",
            "users": val_stage["binary_matrix"].shape[0],
            "items": val_stage["binary_matrix"].shape[1],
            "eval_users": len(val_stage["truth"]),
        },
        {
            "stage": "test",
            "users": test_stage["binary_matrix"].shape[0],
            "items": test_stage["binary_matrix"].shape[1],
            "eval_users": len(test_stage["truth"]),
        },
    ]
)

,stage,users,items,eval_users
0,validation,2366,161487,2024
1,test,2366,161829,1997


те же функции будем использовать и для candidate generation и для ranking


In [36]:
def recommendations_to_dict(df, top_k):
    if df.empty:
        return {}
    frame = df.sort_values(["uid", "rank", "score"], ascending=[True, True, False]).copy()
    frame = frame[frame["rank"] <= top_k]
    grouped = frame.groupby("uid")["item_id"].agg(list)
    return {int(uid): [int(item) for item in items[:top_k]] for uid, items in grouped.items()}

def recall_at_k(recs, truth, k):
    hits, total = 0, 0
    for uid, true_items in truth.items():
        user_recs = recs.get(uid, [])[:k]
        hits += len(set(user_recs) & true_items)
        total += len(true_items)
    return hits / total if total else 0.0

def precision_at_k(recs, truth, k):
    vals = []
    for uid, true_items in truth.items():
        user_recs = recs.get(uid, [])[:k]
        if user_recs:
            vals.append(len(set(user_recs) & true_items) / k)
    return float(np.mean(vals)) if vals else 0.0

def ndcg_at_k(recs, truth, k):
    vals = []
    for uid, true_items in truth.items():
        ideal = sum(1 / np.log2(i + 2) for i in range(min(len(true_items), k)))
        if ideal == 0:
            continue
        dcg = 0.0
        for i, item_id in enumerate(recs.get(uid, [])[:k]):
            if item_id in true_items:
                dcg += 1 / np.log2(i + 2)
        vals.append(dcg / ideal)
    return float(np.mean(vals)) if vals else 0.0

def map_at_k(recs, truth, k):
    vals = []
    for uid, true_items in truth.items():
        if not true_items:
            continue
        hits, prec_sum = 0, 0.0
        for rank, item_id in enumerate(recs.get(uid, [])[:k], start=1):
            if item_id in true_items:
                hits += 1
                prec_sum += hits / rank
        vals.append(prec_sum / min(len(true_items), k))
    return float(np.mean(vals)) if vals else 0.0

def hitrate_at_k(recs, truth, k):
    vals = [float(bool(set(recs.get(uid, [])[:k]) & true_items)) for uid, true_items in truth.items()]
    return float(np.mean(vals)) if vals else 0.0

def eval_table(df, truth, ks, name, include_precision=False):
    rows = []
    for k in ks:
        recs = recommendations_to_dict(df, k)
        row = {
            "model": name,
            "k": k,
            "recall": recall_at_k(recs, truth, k),
            "ndcg": ndcg_at_k(recs, truth, k),
            "map": map_at_k(recs, truth, k),
            "hitrate": hitrate_at_k(recs, truth, k),
        }
        if include_precision:
            row["precision"] = precision_at_k(recs, truth, k)
        rows.append(row)
    return pd.DataFrame(rows)

рекомендуем самые популярные positive айтемы


In [37]:
def fit_popular(stage):
    scores = np.asarray(stage["binary_matrix"].sum(axis=0)).ravel().astype(np.float32)
    order = np.argsort(scores)[::-1]
    return {"scores": scores, "order": order}

def rec_popular(stage, model, uid, k):
    if uid not in stage["user2idx"]:
        return []

    seen = set(stage["seen_items"][uid].tolist())
    out = []
    for item_idx in model["order"]:
        if int(item_idx) in seen:
            continue
        out.append((int(stage["idx2item"][item_idx]), float(model["scores"][item_idx])))
        if len(out) >= k:
            break
    return out

тут ищем похожих пользователей на positive матрице


In [38]:
def fit_userknn(stage, n_neighbors=100):
    index = NearestNeighbors(
        metric="cosine",
        algorithm="brute",
        n_neighbors=min(n_neighbors + 1, stage["binary_matrix"].shape[0]),
    )
    index.fit(stage["binary_matrix"])
    return index

def rec_userknn(stage, index, uid, k):
    if uid not in stage["user2idx"]:
        return []

    user_idx = stage["user2idx"][uid]
    distances, neighbors = index.kneighbors(stage["binary_matrix"][user_idx])
    scores = np.zeros(stage["binary_matrix"].shape[1], dtype=np.float32)

    for distance, neighbor_idx in zip(distances[0], neighbors[0]):
        if neighbor_idx == user_idx:
            continue

        similarity = 1.0 - distance
        if similarity <= 0:
            continue

        row = stage["binary_matrix"][neighbor_idx]
        scores[row.indices] += similarity * row.data

    scores[stage["seen_items"][uid]] = -np.inf
    best = np.argpartition(scores, -k)[-k:]
    best = best[np.argsort(scores[best])[::-1]]
    return [(int(stage["idx2item"][item_idx]), float(scores[item_idx])) for item_idx in best]

здесь используем weighted таблицу


In [39]:
def fit_als(stage, factors=64, regularization=0.08, iterations=20):
    model = AlternatingLeastSquares(
        factors=factors,
        regularization=regularization,
        iterations=iterations,
        random_state=42,
        num_threads=0,
    )
    model.fit(stage["weighted_matrix"], show_progress=False)
    return model

def rec_als(stage, model, uid, k):
    if uid not in stage["user2idx"]:
        return []

    user_idx = stage["user2idx"][uid]
    item_idx, scores = model.recommend(
        user_idx,
        stage["weighted_matrix"][user_idx],
        N=k,
        filter_already_liked_items=True,
    )

    return [(int(stage["idx2item"][i]), float(score)) for i, score in zip(item_idx.tolist(), scores.tolist())]

приводим все модели к одному формату и делаем RRF blend


In [40]:
def make_rec_frame(stage, rec_fn, name, k):
    rows = []
    for uid in stage["truth"].keys():
        recs = rec_fn(uid, k)
        for rank, (item_id, score) in enumerate(recs, start=1):
            rows.append({"uid": int(uid), "item_id": int(item_id), "score": float(score), "rank": rank, "model": name})
    return pd.DataFrame(rows)

def blend_rrf(frames, k):
    long_df = pd.concat(frames, ignore_index=True)
    long_df["rrf"] = 1.0 / (60.0 + long_df["rank"])

    out = (
        long_df.groupby(["uid", "item_id"], as_index=False)
        .agg(score=("rrf", "sum"), votes=("model", "nunique"), best_rank=("rank", "min"))
        .sort_values(["uid", "score", "votes", "best_rank"], ascending=[True, False, False, True])
    )

    out["rank"] = out.groupby("uid").cumcount() + 1
    out = out[out["rank"] <= k].copy()
    out["model"] = "blend"
    return out[["uid", "item_id", "score", "rank", "model"]]

сравниваем модели и blend на validation


In [51]:
val_pop_model = fit_popular(val_stage)
val_knn_model = fit_userknn(val_stage)
val_als_model = fit_als(val_stage)

val_pop_df = make_rec_frame(val_stage, lambda uid, k: rec_popular(val_stage, val_pop_model, uid, k), "popular", CANDIDATE_K)
val_knn_df = make_rec_frame(val_stage, lambda uid, k: rec_userknn(val_stage, val_knn_model, uid, k), "userknn", CANDIDATE_K)
val_als_df = make_rec_frame(val_stage, lambda uid, k: rec_als(val_stage, val_als_model, uid, k), "als", CANDIDATE_K)
val_blend_df = blend_rrf([val_pop_df, val_knn_df, val_als_df], CANDIDATE_K)

validation_metrics = pd.concat(
    [
        eval_table(val_pop_df, val_stage["truth"], (50, 100), "popular"),
        eval_table(val_knn_df, val_stage["truth"], (50, 100), "userknn"),
        eval_table(val_als_df, val_stage["truth"], (50, 100), "als"),
        eval_table(val_blend_df, val_stage["truth"], (50, 100), "blend"),
    ],
    ignore_index=True,
)

validation_metrics.sort_values(["k", "recall"], ascending=[True, False])

,model,k,recall,ndcg,map,hitrate
4,als,50,0.118083,0.049142,0.032165,0.118083
2,userknn,50,0.113142,0.044987,0.028146,0.113142
6,blend,50,0.108696,0.037735,0.020823,0.108696
0,popular,50,0.044466,0.016855,0.010077,0.044466
3,userknn,100,0.156621,0.052005,0.028755,0.156621
7,blend,100,0.151680,0.044680,0.021427,0.151680
5,als,100,0.150692,0.054355,0.032603,0.150692
1,popular,100,0.066206,0.020395,0.010393,0.066206


в ranking используем:
score rank от candidate models,
user фичи,
item фичи,
признаки по источникам likes dislikes listens,
флаг known negative


In [42]:
def to_wide(long_df, model_names):
    base = long_df[["uid", "item_id"]].drop_duplicates().reset_index(drop=True)

    for name in model_names:
        part = (
            long_df[long_df["model"] == name][["uid", "item_id", "score", "rank"]]
            .rename(columns={"score": f"{name}_score", "rank": f"{name}_rank"})
        )
        base = base.merge(part, on=["uid", "item_id"], how="left")
        base[f"{name}_present"] = base[f"{name}_rank"].notna().astype("int8")
        base[f"{name}_score"] = base[f"{name}_score"].fillna(0.0)
        base[f"{name}_rank"] = base[f"{name}_rank"].fillna(CANDIDATE_K + 1)

    base["candidate_vote_count"] = base[[f"{name}_present" for name in model_names]].sum(axis=1)
    base["candidate_best_rank"] = base[[f"{name}_rank" for name in model_names]].min(axis=1)
    base["candidate_rrf_score"] = 0.0
    for name in model_names:
        base["candidate_rrf_score"] += base[f"{name}_present"] / (60.0 + base[f"{name}_rank"])

    return base

def make_feature_df(stage, long_df, model_names, truth=None):
    base = to_wide(long_df, model_names)

    history_all = stage["history_all"]
    history_pos = stage["history_positive"]

    user_feats = history_all.groupby("uid", as_index=False).agg(
        user_all_pairs=("item_id", "nunique"),
        user_negative_count=("is_negative", "sum"),
        user_like_count=("has_like", "sum"),
        user_dislike_count=("has_dislike", "sum"),
        user_listen_count=("has_listen", "sum"),
        user_strong_listen_count=("strong_listen", "sum"),
        user_mean_weight=("interaction_weighted", "mean"),
    )
    user_pos_feats = history_pos.groupby("uid", as_index=False).agg(
        user_positive_count=("item_id", "nunique"),
    )

    item_feats = history_all.groupby("item_id", as_index=False).agg(
        item_all_users=("uid", "nunique"),
        item_negative_count=("is_negative", "sum"),
        item_like_count=("has_like", "sum"),
        item_dislike_count=("has_dislike", "sum"),
        item_listen_count=("has_listen", "sum"),
        item_strong_listen_count=("strong_listen", "sum"),
        item_mean_weight=("interaction_weighted", "mean"),
    )
    item_pos_feats = history_pos.groupby("item_id", as_index=False).agg(
        item_positive_count=("uid", "nunique"),
    )

    feats = base.merge(user_feats, on="uid", how="left")
    feats = feats.merge(user_pos_feats, on="uid", how="left")
    feats = feats.merge(item_feats, on="item_id", how="left")
    feats = feats.merge(item_pos_feats, on="item_id", how="left")

    negative_pairs = history_all[history_all["is_negative"] == 1][["uid", "item_id"]].drop_duplicates().copy()
    negative_pairs["known_negative"] = 1
    feats = feats.merge(negative_pairs, on=["uid", "item_id"], how="left")
    feats["known_negative"] = feats["known_negative"].fillna(0).astype("int8")

    feats["user_negative_ratio"] = feats["user_negative_count"] / feats["user_all_pairs"].clip(lower=1)
    feats["user_listen_ratio"] = feats["user_listen_count"] / feats["user_all_pairs"].clip(lower=1)
    feats["item_pos_minus_neg"] = feats["item_positive_count"].fillna(0) - feats["item_negative_count"].fillna(0)
    feats["item_like_minus_dislike"] = feats["item_like_count"].fillna(0) - feats["item_dislike_count"].fillna(0)
    feats["pair_pop_vs_user_activity"] = feats["item_positive_count"].fillna(0) / (1.0 + feats["user_positive_count"].fillna(0))
    feats["pair_item_weight_vs_user_mean"] = feats["item_mean_weight"].fillna(0) - feats["user_mean_weight"].fillna(0)

    if truth is not None:
        feats["label"] = [
            int(int(item_id) in truth.get(int(uid), set()))
            for uid, item_id in zip(feats["uid"], feats["item_id"])
        ]

    return feats.fillna(0.0)


пока используем простой baseline LogisticRegression


In [43]:
model_names = ["popular", "userknn", "als"]

train_rank_df = make_feature_df(
    val_stage,
    pd.concat([val_pop_df, val_knn_df, val_als_df], ignore_index=True),
    model_names,
    truth=val_stage["truth"],
)

feature_cols = [col for col in train_rank_df.columns if col not in {"uid", "item_id", "label"}]

ranker = Pipeline(
    [
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)),
    ]
)

ranker.fit(train_rank_df[feature_cols], train_rank_df["label"])

train_rank_df.head()

,uid,item_id,popular_score,popular_rank,popular_present,userknn_score,userknn_rank,userknn_present,als_score,als_rank,als_present,candidate_vote_count,candidate_best_rank,candidate_rrf_score,user_all_pairs,user_negative_count,user_like_count,user_dislike_count,user_listen_count,user_strong_listen_count,user_mean_weight,user_positive_count,item_all_users,item_negative_count,item_like_count,item_dislike_count,item_listen_count,item_strong_listen_count,item_mean_weight,item_positive_count,known_negative,user_negative_ratio,user_listen_ratio,item_pos_minus_neg,item_like_minus_dislike,pair_pop_vs_user_activity,pair_item_weight_vs_user_mean,label
0,700,9378983,502.0,1.0,1,1.397321,5.0,1,0.075247,2.0,1,3,1.0,0.047907,380,0,0,0,380,191,0.189095,191,910,8,31,8,905,486,0.223824,502,0,0.0,1.0,494,23,2.614583,0.034729,0
1,700,3542184,498.0,2.0,1,1.563383,1.0,1,0.079954,1.0,1,3,1.0,0.048916,380,0,0,0,380,191,0.189095,191,995,20,71,20,984,476,0.247153,498,0,0.0,1.0,478,51,2.593750,0.058058,0
2,700,5862961,497.0,3.0,1,1.468597,2.0,1,0.057599,6.0,1,3,2.0,0.047154,380,0,0,0,380,191,0.189095,191,929,9,52,9,924,478,0.241752,497,0,0.0,1.0,488,43,2.588542,0.052658,0
3,700,6901374,492.0,4.0,1,1.329761,7.0,1,0.066752,4.0,1,3,4.0,0.046175,380,0,0,0,380,191,0.189095,191,862,5,46,5,858,471,0.246954,492,0,0.0,1.0,487,41,2.562500,0.057859,0
4,700,2185191,472.0,5.0,1,1.146690,12.0,1,0.032144,20.0,1,3,5.0,0.041774,380,0,0,0,380,191,0.189095,191,800,6,67,6,797,447,0.279839,472,0,0.0,1.0,466,61,2.458333,0.090744,0


Теперь делаю финальную проверку


In [44]:
test_pop_model = fit_popular(test_stage)
test_knn_model = fit_userknn(test_stage)
test_als_model = fit_als(test_stage)

test_pop_df = make_rec_frame(test_stage, lambda uid, k: rec_popular(test_stage, test_pop_model, uid, k), "popular", CANDIDATE_K)
test_knn_df = make_rec_frame(test_stage, lambda uid, k: rec_userknn(test_stage, test_knn_model, uid, k), "userknn", CANDIDATE_K)
test_als_df = make_rec_frame(test_stage, lambda uid, k: rec_als(test_stage, test_als_model, uid, k), "als", CANDIDATE_K)
test_blend_df = blend_rrf([test_pop_df, test_knn_df, test_als_df], CANDIDATE_K)

Смотрим качество первого этапа


In [45]:
test_candidate_metrics = pd.concat(
    [
        eval_table(test_pop_df, test_stage["truth"], (50, 100), "popular"),
        eval_table(test_knn_df, test_stage["truth"], (50, 100), "userknn"),
        eval_table(test_als_df, test_stage["truth"], (50, 100), "als"),
        eval_table(test_blend_df, test_stage["truth"], (50, 100), "blend"),
    ],
    ignore_index=True,
)

test_candidate_metrics.sort_values(["k", "recall"], ascending=[True, False])

,model,k,recall,ndcg,map,hitrate
4,als,50,0.116174,0.047528,0.030487,0.116174
2,userknn,50,0.109664,0.041578,0.024875,0.109664
6,blend,50,0.106159,0.035584,0.018674,0.106159
0,popular,50,0.046570,0.016613,0.009544,0.046570
5,als,100,0.159740,0.054600,0.031111,0.159740
3,userknn,100,0.146720,0.047561,0.025394,0.146720
7,blend,100,0.146720,0.042107,0.019231,0.146720
1,popular,100,0.067601,0.020017,0.009841,0.067601


здесь строим финальный список рекомендаций


In [46]:
test_rank_df = make_feature_df(
    test_stage,
    pd.concat([test_pop_df, test_knn_df, test_als_df], ignore_index=True),
    model_names,
    truth=test_stage["truth"],
)

test_rank_df["score"] = ranker.predict_proba(test_rank_df[feature_cols])[:, 1]

test_rank_df = test_rank_df.sort_values(
    ["uid", "score", "candidate_rrf_score"],
    ascending=[True, False, False],
).copy()

test_rank_df["rank"] = test_rank_df.groupby("uid").cumcount() + 1

final_recommendations = test_rank_df[["uid", "item_id", "score", "rank"]].copy()
final_recommendations = final_recommendations[final_recommendations["rank"] <= 20]

final_recommendations.head()

,uid,item_id,score,rank
13,700,8213481,0.689742,1
41,700,956298,0.650718,2
2,700,3542184,0.616264,3
199702,700,7887169,0.598318,4
82,700,5865638,0.555717,5


In [47]:
ranking_metrics = eval_table(
    final_recommendations,
    test_stage["truth"],
    (10, 20),
    "ranker",
    include_precision=True,
)

ranking_metrics

,model,k,recall,ndcg,map,hitrate,precision
0,ranker,10,0.071607,0.044952,0.036832,0.071607,0.007161
1,ranker,20,0.095143,0.050845,0.038420,0.095143,0.004757


In [48]:
example_uid = int(final_recommendations["uid"].iloc[0])
final_recommendations[final_recommendations["uid"] == example_uid].head(FINAL_K)

,uid,item_id,score,rank
13,700,8213481,0.689742,1
41,700,956298,0.650718,2
2,700,3542184,0.616264,3
199702,700,7887169,0.598318,4
82,700,5865638,0.555717,5
199725,700,3625362,0.546452,6
199701,700,24177,0.545166,7
199723,700,7282794,0.534008,8
15,700,906358,0.532237,9
199732,700,4822087,0.531500,10
